<a href="https://colab.research.google.com/github/T-Chiunzi/Tava-Smart-Finance-Assistant/blob/main/Savings_Assistant_Tester.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Pseudocode

Send greetings to the user

Display Dashboard

Ask the user to input their salary

Ask user what they want to save for and how much of their salary they want to dedicate to it

User inputs their savings goals or 0 to stop

Display to the user that they should input their expenses

Ask the user if they want to input their expenses via CSV or manually.

If via CSV, run it

If manually, ask the user to input expenses or 0 to stop

Read the data

Output table of expenses

Output a pie chart of expenses

If expenses are more than what they want to save, then advise which expenses they should reduce

Ask the user for how long they want to save for their savings goal by months or indefinitely (for emergency savings)

Ask the user if they want to increase the amount they save by (by % or by money)

Display a timeline of savings with visual of a person running along the line


In [ ]:
# Initialize a global dictionary to hold the user's financial profile
user_profile = {
    "name": "",
    "monthly_salary": 0.0,
    "emergency_goal_monthly": 0.0,
    "has_secondary_goal": False,
    "secondary_goal_amount": 0.0,
    "secondary_goal_date": "",
    "total_expenses": 0.0,
    "calculated_savings_capacity": 0.0
}

In [24]:
def process_user_profile(name, salary, emergency_amount, use_secondary, secondary_amount, secondary_date):
    global user_profile

    # Clean up name input
    if not name or name.strip() == "":
        return "⚠️ Validation Error: Please enter your name to set up your profile."

    # 1. Validate Salary (Must be strictly greater than 0)
    try:
        val_salary = float(salary) if salary else 0.0
        if val_salary <= 0:
            return "⚠️ Validation Error: Monthly salary must be a positive number greater than $0.00."
    except ValueError:
        return "⚠️ Validation Error: Please enter a valid number for monthly salary."

    # 2. Validate Emergency Target (Must be strictly greater than 0)
    try:
        val_emergency = float(emergency_amount) if emergency_amount else 0.0
        if val_emergency <= 0:
            return "⚠️ Validation Error: Emergency savings goal must be a positive number greater than $0.00."
    except ValueError:
        return "⚠️ Validation Error: Please enter a valid number for your emergency savings goal."

    # 3. Validate Secondary Goal (If selected, amount must be strictly greater than 0)
    val_secondary = 0.0
    if use_secondary:
        try:
            val_secondary = float(secondary_amount) if secondary_amount else 0.0
            if val_secondary <= 0:
                return "⚠️ Validation Error: Secondary savings goal amount must be a positive number greater than $0.00."
            if not secondary_date or secondary_date.strip() == "":
                return "⚠️ Validation Error: Please provide a target date for your secondary goal."
        except ValueError:
            return "⚠️ Validation Error: Please enter a valid number for your secondary goal amount."

    # If all checks pass, save values into our dictionary memory
    user_profile["name"] = name.strip()
    user_profile["monthly_salary"] = val_salary
    user_profile["emergency_goal_monthly"] = val_emergency
    user_profile["has_secondary_goal"] = use_secondary
    user_profile["secondary_goal_amount"] = val_secondary
    user_profile["secondary_date"] = secondary_date

    # Friendly confirmation message for the UI
    confirmation_msg = f"### 🎉 Profile Saved Successfully!\n" \
                       f"Welcome **{user_profile['name']}**! Your monthly salary of **${user_profile['monthly_salary']:,.2f}** " \
                       f"and emergency savings goal of **${user_profile['emergency_goal_monthly']:,.2f}/month** have been safely recorded."

    if use_secondary:
        confirmation_msg += f"\n\nWe are also tracking your secondary goal of **${user_profile['secondary_goal_amount']:,.2f}** by **{secondary_date}**."

    return confirmation_msg

In [25]:
import pandas as pd
from datetime import datetime

def calculate_savings(manual_expenses, csv_file):
    global user_profile

    # First, check if a profile exists by testing salary
    if user_profile["monthly_salary"] <= 0:
        return "⚠️ System Error: No active user profile found. Please set up and save your profile in Tab 1 first!"

    # 1. Determine total expenses and apply constraints
    if csv_file is not None:
        try:
            df = pd.read_csv(csv_file.name)
            if 'Amount' in df.columns:
                total_expenses = df['Amount'].abs().sum()
            elif 'Cost' in df.columns:
                total_expenses = df['Cost'].abs().sum()
            else:
                numeric_cols = df.select_dtypes(include=['number']).columns
                if len(numeric_cols) > 0:
                    total_expenses = df[numeric_cols[0]].abs().sum()
                else:
                    return "⚠️ CSV Error: Could not find any numeric expense column in your file."

            # Constraint for empty or zero-value CSV data
            if total_expenses <= 0:
                return "⚠️ CSV Error: The uploaded file contains no transactions or total expenses equal $0.00."

        except Exception as e:
            return f"⚠️ Error processing CSV file: {str(e)}"
    else:
        # Validate manual input constraints
        try:
            total_expenses = float(manual_expenses) if manual_expenses else 0.0
            if total_expenses <= 0:
                return "⚠️ Validation Error: Monthly expenses must be a positive number greater than $0.00."
        except ValueError:
            return "⚠️ Validation Error: Please enter a valid number for your monthly expenses."

    # Save total expenses to our memory dictionary
    user_profile["total_expenses"] = total_expenses

    # 2. Financial Math Calculations
    salary = user_profile["monthly_salary"]
    leftover_income = salary - total_expenses
    user_profile["calculated_savings_capacity"] = leftover_income

    emergency_target = user_profile["emergency_goal_monthly"]
    total_needed_monthly = emergency_target

    secondary_details = ""
    if user_profile["has_secondary_goal"] and user_profile["secondary_goal_amount"] > 0:
        sec_amount = user_profile["secondary_goal_amount"]
        sec_date_str = user_profile["secondary_date"]

        months_remaining = 1
        try:
            target_date = datetime.strptime(sec_date_str, "%Y-%m-%d")
            current_date = datetime.now()
            months_remaining = (target_date.year - current_date.year) * 12 + target_date.month - current_date.month
            if months_remaining <= 0:
                months_remaining = 1
        except:
            pass

        sec_monthly_needed = sec_amount / months_remaining
        total_needed_monthly += sec_monthly_needed
        secondary_details = f"* **Secondary Goal Monthly Target:** ${sec_monthly_needed:,.2f} (To reach ${sec_amount:,.2f} by {sec_date_str})\n"

    # 3. Generate Report String
    report = f"## 📊 Financial Analysis Report for {user_profile['name']}\n" \
             f"---\n" \
             f"* **Total Income:** ${salary:,.2f}\n" \
             f"* **Total Logged Expenses:** ${total_expenses:,.2f}\n" \
             f"* **Remaining Cash Flow:** ${leftover_income:,.2f}\n\n" \
             f"### 🎯 Savings Commitments:\n" \
             f"* **Emergency Fund Target:** ${emergency_target:,.2f}/month\n" \
             f"{secondary_details}" \
             f"* **Total Ideal Monthly Savings:** ${total_needed_monthly:,.2f}\n" \
             f"---\n"

    if leftover_income >= total_needed_monthly:
        report += f"### ✅ Status: On Track!\n" \
                  f"Excellent job! You have enough cash flow to cover your goals and still have " \
                  f"**${leftover_income - total_needed_monthly:,.2f}** remaining for leisurely spending."
    elif leftover_income >= emergency_target:
        report += f"### ⚠️ Status: Partial Progress\n" \
                  f"You can comfortably cover your Emergency Fund, but you are short by " \
                  f"**${total_needed_monthly - leftover_income:,.2f}** to hit your secondary goal target simultaneously. Consider reducing non-essential spending."
    else:
        report += f"### ❌ Status: Budget Deficit\n" \
                  f"Warning! Your current monthly expenses do not leave you enough room to hit your baseline savings goals. " \
                  f"You are short by **${total_needed_monthly - leftover_income:,.2f}** this month. Ask your Chat Coach for budgeting tips!"

    return report

In [26]:
print("--- TEST CASE A: Testing Invalid Profile Input (Zero Salary) ---")
test_a = process_user_profile("Alex", 0, 500, False, 0, "")
print(test_a) # Should output a validation error

print("\n" + "="*50 + "\n")

print("--- TEST CASE B: Testing Invalid Expense Input (Negative Expenses) ---")
# Setup a proper profile first
process_user_profile("Alex", 5000, 500, False, 0, "")
# Test calculator with bad data
test_b = calculate_savings(-1500, None)
print(test_b) # Should output a validation error

--- TEST CASE A: Testing Invalid Profile Input (Zero Salary) ---
⚠️ Validation Error: Monthly salary must be a positive number greater than $0.00.


--- TEST CASE B: Testing Invalid Expense Input (Negative Expenses) ---
⚠️ Validation Error: Monthly expenses must be a positive number greater than $0.00.
